In [ ]:
using Pkg
Pkg.activate(".")
Pkg.status()

In [ ]:
using SymBoltz
M = ΛCDM()

In [ ]:
equations(M.G)

In [ ]:
p = Dict(
  M.g.h => 0.70, M.γ.T₀ => 2.7, M.c.Ω₀ => 0.27, M.b.Ω₀ => 0.05, M.ν.Neff => 3.0,
  M.b.YHe => 0.25, M.h.m_eV => 0.06, M.I.ns => 0.95, M.I.ln_As1e10 => 3.04
)
prob = CosmologyProblem(M, p)

In [ ]:
ks = range(1.0, 1000.0, length=500) # k / (H₀/c)
sol = solve(prob, ks)

In [ ]:
using Plots
plot(sol, log10(M.g.a), [M.b.Xe, M.b.rec.XH⁺, M.b.rec.XHe⁺, M.b.rec.XHe⁺⁺]; grid = false, Nextra = 5, legend_position = :left, framestyle = :box)
savefig("plot1.pdf")

In [ ]:
ks_plot = [1.0, 10.0, 100.0, 1000.0]
plot(sol, log10(M.g.a), [M.g.Φ, M.g.Ψ], ks_plot; grid = false, Nextra = 3, framestyle = :box)
savefig("plot2.pdf")

In [ ]:
plot(sol, log10(M.g.a), [M.ST, 100*M.SE_kχ²], [1.0, 10.0, 100.0, 1000.0], xlims = (-3.3, -2.5), Nextra = 20, grid = false, legend_position = :topright, framestyle = :box)
savefig("plot3.pdf")

In [ ]:
ks = 10 .^ range(0, 3, length=100)
Ps = spectrum_matter([:m, :cb, :h], prob, ks)

In [ ]:
plot(log10.(ks*SymBoltz.k0), log10.(transpose(Ps)/SymBoltz.k0^3), xlabel = "k/(Mpc/h)", ylabel = "P/(h/Mpc)³", label = nothing, ylims = (2, 4.5))

In [ ]:
jl = SphericalBesselCache(25:25:2500)
Cls = spectrum_cmb([:TT, :EE, :ψψ], prob, jl)

In [ ]:
plot(log10.(jl.l), jl.l .^ 2 .* Cls, xlabel = "log10(ℓ)", ylabel = "ℓ² Cᵀᵀ(ℓ)", label = nothing)

In [ ]:
g, τ, k = M.g, M.τ, M.k
a, ℋ, Φ, Ψ = g.a, g.ℋ, g.Φ, g.Ψ
D = Differential(τ)
@parameters w₀ wₐ cₛ² Ω₀ ρ₀=3Ω₀/8π
@variables ρ(τ) P(τ) w(τ) cₐ²(τ) δ(τ,k) θ(τ,k) σ(τ,k)
eqs = [
  w ~ w₀ + wₐ*(1-a)
  ρ ~ ρ₀ * a^(-3(1+w₀+wₐ)) * exp(-3wₐ*(1-a))
  P ~ w * ρ
  cₐ² ~ w - 1/(3ℋ) * D(w)/(1+w)
  D(δ) ~ 3ℋ*(w-cₛ²)*δ - (1+w) * ((1+9(ℋ/k)^2*(cₛ²-cₐ²))*θ - 3*D(Φ))
  D(θ) ~ (3cₛ²-1)*ℋ*θ + k^2*cₛ²*δ/(1+w) + k^2*Ψ
  σ ~ 0
]
initialization_eqs = [
  δ ~ -3//2 * (1+w) * Ψ
  θ ~ 1//2 * (k^2*τ) * Ψ
]
X = System(eqs, τ; initialization_eqs, name = :X)

In [ ]:
M = ΛCDM(Λ = X, name = :w₀wₐCDM)
push!(p, X.w₀ => -0.9, X.wₐ => 0.2, X.cₛ² => 1.0)
prob = CosmologyProblem(M, p)